# Gemma 3 12B Legal AI — Production Training

**Auto-Run Compatible** — Run All (Ctrl+F9) executes top-to-bottom with zero interaction.

| Setting | Value |
|---------|-------|
| Model | `unsloth/gemma-3-12b-it-unsloth-bnb-4bit` (12B, multimodal) |
| GPU | A100 40GB required |
| Datasets | ~30K examples (trimmed from 81K — legal + Svelte 5 + codebase) |
| Training | ~2-3 hours (batch 8, no packing, 2 epochs) |
| Deploy | RTX 3060 Ti via Q4_K_M GGUF → Ollama |

**Note**: Gemma 3 12B is always multimodal (vision + text). Packing is auto-disabled by Unsloth.
We compensate with larger batch size (8) and trimmed dataset (30K).

**Phase 1** (this notebook): Legal + code knowledge, vision frozen, ~2-3h
**Phase 2** (cells 30-35): Emotion + personality, vision unfrozen, ~3-4h

**One-time setup**: Add your HuggingFace token to Colab Secrets (sidebar lock icon → key: `HF_TOKEN`)

## 1. Install + Imports

In [ ]:
# Install Unsloth (skips if already installed — safe for Run All)
import importlib.util, subprocess, sys

if importlib.util.find_spec("unsloth") is None:
    print("Installing Unsloth (first run only)...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
        "--upgrade", "--force-reinstall", "--no-cache-dir", "--no-deps",
        "unsloth", "unsloth_zoo"])
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
        "--upgrade", "--no-cache-dir",
        "bitsandbytes", "accelerate", "peft", "trl", "transformers",
        "datasets", "huggingface_hub", "pillow"])
    print("Install complete — restarting runtime...")
    import os; os._exit(0)  # Clean restart, Colab re-runs from top
else:
    print("Unsloth already installed — skipping")

In [ ]:
# Imports + Config + GPU Check
import os, sys, json, glob
from pathlib import Path

# ── User Config ──────────────────────────────────
USE_WANDB = False  # Set True if you have a W&B account
HF_CODEBASE_REPO = "Semaj90/deeds-legal-codebase"
# ─────────────────────────────────────────────────

if not USE_WANDB:
    os.environ["WANDB_DISABLED"] = "true"
    os.environ["WANDB_MODE"] = "disabled"
    os.environ["DISABLE_MLFLOW_INTEGRATION"] = "true"

# Prevent Colab restart loops
for x in [k for k in sys.modules if "PIL" in k or "google" in k]:
    sys.modules.pop(x, None)

import torch
from unsloth import FastVisionModel, is_bfloat16_supported, get_chat_template
from transformers import TextStreamer
from trl import SFTTrainer, SFTConfig
from datasets import load_dataset, concatenate_datasets, Dataset

# GPU check
assert torch.cuda.is_available(), "No GPU! Runtime -> Change runtime type -> A100"
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"GPU: {gpu_name} ({vram_gb:.0f}GB)")
if "T4" in gpu_name:
    print("WARNING: T4 only 16GB — may OOM. Switch to A100.")
print(f"PyTorch {torch.__version__} | BF16: {is_bfloat16_supported()} | wandb: {'ON' if USE_WANDB else 'OFF'}")

## 2. Model Configuration

In [ ]:
MODEL_NAME = "unsloth/gemma-3-12b-it-unsloth-bnb-4bit"
MAX_SEQ_LENGTH = 512   # 2x faster than 2048, most legal text fits
TRAIN_SIZE = 30000     # Trim dataset (30K is plenty for domain knowledge)

LORA_R = 16            # Rank 16 optimal for 12B
LORA_ALPHA = 32        # 2x rank ratio
LORA_DROPOUT = 0.1

FINETUNE_VISION_LAYERS = False   # SigLIP encoder FROZEN (Phase 1)
FINETUNE_LANGUAGE_LAYERS = True
FINETUNE_ATTENTION_MODULES = True
FINETUNE_MLP_MODULES = True

print(f"Model: {MODEL_NAME}")
print(f"Seq: {MAX_SEQ_LENGTH} | LoRA r={LORA_R} alpha={LORA_ALPHA} | Vision: FROZEN")
print(f"Dataset: {TRAIN_SIZE:,} examples | Batch 8×2=16 effective | ~2-3h on A100")

## 3. Load Datasets

In [ ]:
# HuggingFace auth (auto-run safe: Colab Secrets -> env var -> interactive)
from huggingface_hub import login

hf_token = None
try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
    print("HF token from Colab Secrets")
except Exception:
    hf_token = os.environ.get("HF_TOKEN")
    if hf_token: print("HF token from env")

if not hf_token:
    print("No HF_TOKEN found. Set: Colab sidebar -> Secrets -> HF_TOKEN")
    from huggingface_hub import notebook_login
    notebook_login()
else:
    login(token=hf_token, add_to_git_credential=False)

# Load codebase (private dataset)
print(f"\nLoading: {HF_CODEBASE_REPO}")
try:
    codebase_hf = load_dataset(HF_CODEBASE_REPO, split="train")
    codebase_patterns = [{"text": ex["text"]} for ex in codebase_hf]
    print(f"Codebase: {len(codebase_patterns):,} examples")
except Exception as e:
    print(f"HF failed ({e}), falling back to Google Drive...")
    from google.colab import drive
    drive.mount("/content/drive")
    dataset_dir = None
    for p in [Path("/content/drive/MyDrive/COLAB_PACKAGE/COLAB_PACKAGE/training-datasets"),
              Path("/content/drive/MyDrive/COLAB_PACKAGE/training-datasets")]:
        if p.exists(): dataset_dir = p; break
    assert dataset_dir, "No training data! Upload to HF or Google Drive."
    codebase_patterns = []
    for f in sorted(dataset_dir.glob("*.jsonl")):
        if "-old" in f.name or f.name.startswith("."): continue
        for line in open(f, encoding="utf-8"):
            if line.strip():
                try: codebase_patterns.append(json.loads(line))
                except: pass
    print(f"Codebase: {len(codebase_patterns):,} examples (Google Drive)")

In [ ]:
# Load 8 public legal + code datasets
from unsloth.chat_templates import standardize_data_formats

def standardize_text(example):
    if 'text' not in example: return example
    if isinstance(example['text'], list):
        example['text'] = ' '.join(
            item['value'] if isinstance(item, dict) and 'value' in item else str(item)
            for item in example['text'])
    elif not isinstance(example['text'], str):
        example['text'] = str(example['text'])
    return example

datasets_config = [
    ("FineTome",    "mlabonne/FineTome-100k",             None,        "train[:10000]"),
    ("GSM8K",       "openai/gsm8k",                       "main",      "train[:5000]"),
    ("Pile of Law", "lamblamb/pile_of_law_subset",         None,        "train[:25000]"),
    ("LEDGAR",      "lex_glue",                            "ledgar",    "train[:10000]"),
    ("Case Hold",   "lighteval/lexglue",                   "case_hold", "train[:5000]"),
    ("SCOTUS",      "lighteval/lexglue",                   "scotus",    "train[:5000]"),
    ("Svelte 5",    "Dreamslol/svelte-5-sveltekit-2",     None,        "train"),
    ("CAP",         "common-pile/caselaw_access_project",  None,        "train[:10000]"),
]

legal_datasets = []
for i, (name, repo, config, split) in enumerate(datasets_config, 1):
    print(f"[{i}/8] {name}...", end=" ")
    ds = load_dataset(repo, config, split=split) if config else load_dataset(repo, split=split)
    ds = standardize_data_formats(ds)
    # Find and rename text column
    for col in ['text', 'conversations', 'instruction', 'question', 'input', 'content']:
        if col in ds.column_names:
            if col != 'text': ds = ds.rename_column(col, 'text')
            break
    if 'text' in ds.column_names:
        ds = ds.select_columns(['text']).map(standardize_text, num_proc=4)
        legal_datasets.append(ds)
        print(f"{len(ds):,}")
    else:
        print(f"SKIPPED ({ds.column_names})")

legal_dataset = concatenate_datasets(legal_datasets)
print(f"\nPublic datasets total: {len(legal_dataset):,}")

## 4. Combine + Format for Chat

In [ ]:
# Combine all datasets
codebase_dataset = Dataset.from_list(codebase_patterns)
combined_dataset = concatenate_datasets([legal_dataset, codebase_dataset])
print(f"Combined: {len(combined_dataset):,} (public: {len(legal_dataset):,} + codebase: {len(codebase_dataset):,})")

In [ ]:
# Format as ShareGPT conversations for Gemma 3 chat template
from unsloth.chat_templates import standardize_sharegpt

def format_for_chat(example):
    text = example.get('text', '')
    if not text or not isinstance(text, str):
        return {"conversations": [{"from": "human", "value": "Explain:"}, {"from": "gpt", "value": "N/A"}]}
    # Auto-detect instruction type
    kw_map = [
        (['$state', '$derived', '$effect', '$props', 'runes', 'svelte 5'], 'Explain this Svelte 5 runes pattern:'),
        (['sveltekit', '+page.svelte', '+server.ts', 'load function'], 'Explain this SvelteKit pattern:'),
        (['evidence', 'forensic', 'rag', 'upload'], 'Explain this legal evidence processing concept:'),
        (['statute', 'citation', 'u.s.c', 'case law'], 'Explain this legal citation or statute:'),
        (['tensorrt', 'triton', 'trt-llm', 'onnx'], 'Explain this AI inference deployment concept:'),
        (['typescript', 'type', 'interface'], 'Explain this TypeScript pattern:'),
        (['svelte', 'component', '.svelte'], 'Explain this Svelte programming pattern:'),
    ]
    instruction = 'Explain the following concept:'
    lower = text.lower()
    for keywords, instr in kw_map:
        if any(kw in lower for kw in keywords):
            instruction = instr
            break
    return {"conversations": [{"from": "human", "value": instruction}, {"from": "gpt", "value": text}]}

print("Formatting for ShareGPT...")
train_dataset = combined_dataset.map(format_for_chat, remove_columns=['text'], num_proc=4)
train_dataset = standardize_sharegpt(train_dataset)

# Debug: show actual conversation structure
sample = train_dataset[0]['conversations']
print(f"Conversation keys: {list(sample[0].keys())}")
print(f"Sample: {str(sample[0])[:120]}...")

# Filter out empty/null conversations
pre_filter = len(train_dataset)
train_dataset = train_dataset.filter(
    lambda ex: any(
        len((c.get("value") or c.get("content") or "").strip()) > 0
        for c in ex["conversations"]
    ),
    num_proc=4,
)
print(f"{len(train_dataset):,} examples (dropped {pre_filter - len(train_dataset):,} empty)")

# Trim to TRAIN_SIZE for faster training (Gemma 3 12B can't use packing)
if len(train_dataset) > TRAIN_SIZE:
    train_dataset = train_dataset.shuffle(seed=42).select(range(TRAIN_SIZE))
    print(f"Trimmed to {len(train_dataset):,} examples (from {pre_filter:,})")
else:
    print(f"Using all {len(train_dataset):,} examples")

## 5. Load Model + LoRA

In [ ]:
# Load Gemma 3 12B (4-bit quantized)
# Gemma 3 12B is always multimodal — packing auto-disabled by Unsloth
# We compensate with batch 8 + trimmed dataset
print(f"Loading {MODEL_NAME}...")
model, tokenizer = FastVisionModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    dtype=None,
)
tokenizer = get_chat_template(tokenizer, chat_template="gemma-3")
print(f"Loaded | Chat template: Gemma 3 | Packing: OFF (vision model)")

In [ ]:
# Add LoRA adapters (vision FROZEN for Phase 1)
model = FastVisionModel.get_peft_model(
    model,
    r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
    finetune_vision_layers=FINETUNE_VISION_LAYERS,
    finetune_language_layers=FINETUNE_LANGUAGE_LAYERS,
    finetune_attention_modules=FINETUNE_ATTENTION_MODULES,
    finetune_mlp_modules=FINETUNE_MLP_MODULES,
    use_gradient_checkpointing="unsloth",
    use_rslora=True,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    random_state=42,
)
model.print_trainable_parameters()

## 6. Train

In [ ]:
# Training arguments — optimized for speed (batch 8, no packing, checkpoints every 500 steps)
training_args = SFTConfig(
    output_dir="./gemma3-12b-legal-outputs",
    num_train_epochs=2,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    warmup_steps=50,
    fp16=False, bf16=True, bf16_full_eval=True,
    logging_steps=25,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=2,  # Keep last 2 checkpoints (saves disk)
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    max_grad_norm=1.0,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": True},
    dataloader_num_workers=4,
    dataloader_pin_memory=True,
    seed=42, data_seed=42,
    report_to="none",
    packing=False,
    max_seq_length=MAX_SEQ_LENGTH,
)
print(f"Batch: {training_args.per_device_train_batch_size}x{training_args.gradient_accumulation_steps}={training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps} effective | LR: {training_args.learning_rate} cosine | Epochs: {training_args.num_train_epochs}")
print(f"Packing: OFF (Gemma 3 12B is multimodal) | Checkpoints: every {training_args.save_steps} steps (keep {training_args.save_total_limit})")

In [ ]:
# Initialize trainer + mask user prompts (train on assistant responses only)
from unsloth.chat_templates import train_on_responses_only

def formatting_prompts_func(examples):
    convos = examples["conversations"]
    if isinstance(convos, list) and len(convos) > 0 and isinstance(convos[0], dict):
        return [tokenizer.apply_chat_template(convos, tokenize=False, add_generation_prompt=False)]
    return [tokenizer.apply_chat_template(c, tokenize=False, add_generation_prompt=False) for c in convos]

trainer = SFTTrainer(
    model=model, tokenizer=tokenizer, train_dataset=train_dataset,
    args=training_args,
    formatting_func=formatting_prompts_func,
)

# Mask user prompts — only train on assistant (model) outputs
# This significantly improves accuracy by not wasting gradient updates on inputs
trainer = train_on_responses_only(
    trainer,
    instruction_part="<start_of_turn>user\n",
    response_part="<start_of_turn>model\n",
)

# Verify masking works — show what the model actually trains on
sample_tokens = trainer.train_dataset[100]["input_ids"]
sample_labels = trainer.train_dataset[100]["labels"]
print("Chat template check (decoded input):")
print(tokenizer.decode(sample_tokens)[:200] + "...")
print("\nMasked training target (only assistant response visible):")
masked = tokenizer.decode([tokenizer.pad_token_id if x == -100 else x for x in sample_labels])
print(masked.replace(tokenizer.pad_token, " ")[:200] + "...")
print(f"\nTrainer ready | Responses-only training | Seq: {MAX_SEQ_LENGTH}")

In [ ]:
# Train (auto-resumes from checkpoint if one exists)
checkpoint_dirs = sorted(glob.glob("./gemma3-12b-legal-outputs/checkpoint-*"))
if checkpoint_dirs:
    print(f"Resuming from: {checkpoint_dirs[-1]}")
    trainer_stats = trainer.train(resume_from_checkpoint=True)
else:
    print(f"Starting fresh — {len(train_dataset):,} examples, {int(training_args.num_train_epochs)} epochs, responses-only training")
    trainer_stats = trainer.train()

rt = trainer_stats.metrics['train_runtime']
print(f"\nDone: {rt/3600:.1f}h | {trainer_stats.metrics['train_samples_per_second']:.2f} samples/sec")

## 7. Test Inference

In [ ]:
FastVisionModel.for_inference(model)
text_streamer = TextStreamer(tokenizer, skip_prompt=True)

for prompt in [
    "Explain evidence type detection in a legal AI system.",
    "What are Svelte 5 runes?",
    "Describe the RAG evidence upload pipeline.",
]:
    print(f"\n{'='*70}\nPrompt: {prompt}\n{'='*70}")
    inputs = tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}],
        tokenize=True, add_generation_prompt=True, return_tensors="pt",
    ).to("cuda")
    model.generate(input_ids=inputs, streamer=text_streamer,
                   max_new_tokens=256, temperature=0.7, top_p=0.9, use_cache=True)
    print()

## 8. Save + Export

In [ ]:
# Save LoRA adapters
model.save_pretrained("gemma3-12b-legal-lora")
tokenizer.save_pretrained("gemma3-12b-legal-lora")
print("LoRA saved: gemma3-12b-legal-lora/ (~500 MB)")

In [ ]:
# Export merged 16-bit in shards (resumable — if Colab disconnects, re-run this cell)
import shutil

EXPORT_DIR = Path("gemma3-12b-legal-merged-16bit")

# Disk check — merged 16-bit needs ~24GB free
disk_free_gb = shutil.disk_usage("/").free / (1024**3)
print(f"Disk free: {disk_free_gb:.1f} GB (need ~24 GB for merged shards)")
if disk_free_gb < 25:
    # Clean up checkpoints first to free disk
    print("Low disk — cleaning training checkpoints to free space...")
    for ckpt in sorted(glob.glob("./gemma3-12b-legal-outputs/checkpoint-*")):
        shutil.rmtree(ckpt, ignore_errors=True)
        print(f"  Deleted: {ckpt}")
    disk_free_gb = shutil.disk_usage("/").free / (1024**3)
    print(f"Disk free after cleanup: {disk_free_gb:.1f} GB")
    if disk_free_gb < 20:
        print("WARNING: Still low on disk. Export may fail. Consider mounting Google Drive first.")

print(f"\n[1/2] Saving merged 16-bit shards...")
model.save_pretrained_merged(
    str(EXPORT_DIR), tokenizer, save_method="merged_16bit",
)

shard_files = sorted(EXPORT_DIR.glob("model-*.safetensors"))
if not shard_files:
    shard_files = sorted(EXPORT_DIR.glob("*.safetensors"))

other_files = [f for f in EXPORT_DIR.iterdir() if f.suffix != '.safetensors']
total_size = sum(f.stat().st_size for f in EXPORT_DIR.iterdir()) / (1024**3)
print(f"\n[2/2] Export complete:")
print(f"  Safetensor shards: {len(shard_files)}")
print(f"  Other files: {len(other_files)} (config, tokenizer, etc.)")
print(f"  Total: {total_size:.1f} GB")
for f in shard_files:
    print(f"    {f.name}: {f.stat().st_size / (1024**3):.1f} GB")

In [ ]:
# Copy shards to Google Drive (resumable — re-run if Colab disconnects)
# Each shard copies independently. Already-copied shards are SKIPPED (cache-safe).
import time

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

DRIVE_DIR = Path("/content/drive/MyDrive/gemma3-12b-legal-shards")
DRIVE_DIR.mkdir(parents=True, exist_ok=True)

EXPORT_DIR = Path("gemma3-12b-legal-merged-16bit")
all_files = sorted(EXPORT_DIR.iterdir())

print(f"Copying {len(all_files)} files to Google Drive (skips existing)...\n")
copied, skipped, total_bytes = 0, 0, 0

for i, src in enumerate(all_files, 1):
    dst = DRIVE_DIR / src.name
    src_size = src.stat().st_size
    size_gb = src_size / (1024**3)

    # Skip if already copied with matching size
    if dst.exists() and dst.stat().st_size == src_size:
        print(f"  [{i}/{len(all_files)}] SKIP {src.name} ({size_gb:.1f} GB) — already on Drive")
        skipped += 1
        continue

    # Copy with progress
    print(f"  [{i}/{len(all_files)}] Copying {src.name} ({size_gb:.1f} GB)...", end=" ", flush=True)
    t0 = time.time()
    shutil.copy2(str(src), str(dst))
    elapsed = time.time() - t0
    speed = src_size / elapsed / (1024**2)  # MB/s
    print(f"done ({elapsed:.0f}s, {speed:.0f} MB/s)")
    copied += 1
    total_bytes += src_size

# Verify all files made it
drive_files = list(DRIVE_DIR.iterdir())
print(f"\n{'='*60}")
print(f"Results: {copied} copied, {skipped} skipped (already existed)")
print(f"Total on Drive: {len(drive_files)} files in {DRIVE_DIR}")
print(f"Copied this run: {total_bytes / (1024**3):.1f} GB")

# List shards for download reference
print(f"\nDownload each shard individually from Google Drive:")
print(f"   https://drive.google.com/drive/folders/")
for f in sorted(DRIVE_DIR.glob("model-*.safetensors")):
    print(f"   - {f.name} ({f.stat().st_size / (1024**3):.1f} GB)")
for f in sorted(DRIVE_DIR.iterdir()):
    if not f.name.startswith("model-"):
        print(f"   - {f.name} ({f.stat().st_size / 1024:.0f} KB)")

print(f"\nIf Colab disconnects mid-copy, just re-run this cell.")
print(f"Already-copied shards will be skipped automatically.")

## 9. Deployment Pipeline — RTX 3060 Ti (Ampere SM 8.6)

**After training completes**, follow this pipeline to deploy the model on your local RTX 3060 Ti via WSL2 + Docker.

**GPU**: RTX 3060 Ti 8GB GDDR6 | Ampere SM 8.6 | 4864 CUDA Cores | 448 GB/s bandwidth

**Two deployment paths** — start with Path A, upgrade to Path B if you need 2x speed:

| | Path A: Ollama Q4_K_M | Path B: TensorRT INT4 + Triton |
|---|---|---|
| **Speed** | 60-70 tok/s | 120-150 tok/s |
| **VRAM** | 7.2 GB | 6.5 GB |
| **Setup** | 5 minutes | 2-4 hours |
| **Context** | 4096 tokens | 2048 tokens |
| **Recommendation** | START HERE | Upgrade if needed |

**Full deployment docs** (in this repo):
- `scripts/unsloth-training/RTX_3060_TI_TRT_BUILD.md` — Both paths A & B
- `scripts/unsloth-training/TENSORRT_TRITON_DEPLOYMENT.md` — 7-phase TRT-LLM + Triton guide
- `scripts/unsloth-training/DEPLOYMENT_ROADMAP.md` — 5-step overview

### Path A: Ollama Q4_K_M (Recommended — 5 min setup)

Run these commands in **WSL2 terminal** on your Windows 10 machine after downloading the shards from Google Drive.

```bash
# ── Step 1: Download shards from Google Drive to Windows ──
# Download each file individually from:
#   Google Drive → gemma3-12b-legal-shards/
# Then copy from Windows Downloads to WSL2:
mkdir -p ~/gemma3-12b-legal/merged-16bit
cp /mnt/c/Users/james/Downloads/model-*.safetensors ~/gemma3-12b-legal/merged-16bit/
cp /mnt/c/Users/james/Downloads/config.json ~/gemma3-12b-legal/merged-16bit/
cp /mnt/c/Users/james/Downloads/tokenizer* ~/gemma3-12b-legal/merged-16bit/
cp /mnt/c/Users/james/Downloads/special_tokens_map.json ~/gemma3-12b-legal/merged-16bit/ 2>/dev/null

# Verify all files present
ls -lh ~/gemma3-12b-legal/merged-16bit/

# ── Step 2: Install llama.cpp + convert to GGUF ──
cd ~/gemma3-12b-legal
git clone https://github.com/ggerganov/llama.cpp && cd llama.cpp
make LLAMA_CUDA=1
python convert-hf-to-gguf.py ~/gemma3-12b-legal/merged-16bit

# ── Step 3: Quantize 24GB FP16 → 7.2GB Q4_K_M ──
./llama-quantize ~/gemma3-12b-legal/merged-16bit/*.gguf gemma3-12b-legal-Q4_K_M.gguf Q4_K_M
ls -lh gemma3-12b-legal-Q4_K_M.gguf  # Expected: ~7.2GB

# ── Step 4: Import to Ollama ──
cat > Modelfile <<'EOF'
FROM ./gemma3-12b-legal-Q4_K_M.gguf
SYSTEM """You are a legal AI assistant trained on Svelte 5, SvelteKit 2,
and legal domain knowledge. You understand evidence analysis, legal reasoning,
and modern web development."""
PARAMETER temperature 0.7
PARAMETER top_p 0.9
PARAMETER repeat_penalty 1.1
PARAMETER num_ctx 4096
PARAMETER num_gpu 999
EOF

ollama create gemma3-12b-legal:latest -f Modelfile
ollama run gemma3-12b-legal:latest "Explain legal evidence types"
# Expected: 60-70 tokens/sec on RTX 3060 Ti, 7.2GB VRAM

# ── Step 5: Verify + wire to SvelteKit ──
nvidia-smi  # Should show ~7.2GB VRAM usage

# Replace existing model tag so SvelteKit picks it up automatically
ollama cp gemma3-12b-legal:latest gemma3-legal:latest
```

**Done!** Your SvelteKit app already uses `gemma3-legal:latest` via Ollama on port 11434 — no code changes needed.

### Path B: TensorRT-LLM INT4 + Triton (Advanced — 2x faster)

Only proceed if Path A's 60-70 tok/s is insufficient. Requires WSL2 + Docker + NVIDIA Container Toolkit.

**Phase 1: Download shards to WSL2 (resumable)**
```bash
# Create model directory
mkdir -p ~/gemma3-12b-legal/merged-16bit && cd ~/gemma3-12b-legal/merged-16bit

# Download each shard individually from Google Drive
# If a download fails, just re-download that ONE shard — others are fine
# Use gdown (pip install gdown) or browser download + WSL2 copy:
#   cp /mnt/c/Users/james/Downloads/model-00001-of-00005.safetensors .
#   cp /mnt/c/Users/james/Downloads/model-00002-of-00005.safetensors .
#   ... etc for all shards + config.json + tokenizer files

# Verify all shards present
ls -lh *.safetensors
# Expected: 5 files × ~5GB each = ~24GB total

# Verify model loads
python -c "
from transformers import AutoModelForCausalLM, AutoTokenizer
model = AutoModelForCausalLM.from_pretrained('.', device_map='cpu', torch_dtype='float16')
print(f'Model: {model.config.model_type} | Params: {sum(p.numel() for p in model.parameters()):,}')
tokenizer = AutoTokenizer.from_pretrained('.')
print(f'Vocab: {tokenizer.vocab_size:,}')
"
```

**Phase 2: TRT-LLM checkpoint conversion (~30-45 min)**
```bash
# Set CUDA arch for RTX 3060 Ti (Ampere SM 8.6)
export CUDA_VISIBLE_DEVICES=0
export TORCH_CUDA_ARCH_LIST="8.6"
export CUDA_LAUNCH_BLOCKING=0

# Clone + install TensorRT-LLM
cd ~/gemma3-12b-legal
git clone https://github.com/NVIDIA/TensorRT-LLM.git
cd TensorRT-LLM
pip install -r requirements.txt && pip install tensorrt_llm

# Convert HF shards → TRT-LLM checkpoint (INT4 quantized)
python examples/gemma/convert_checkpoint.py \
  --model_dir ~/gemma3-12b-legal/merged-16bit \
  --output_dir ./trt_checkpoints/gemma3-12b-legal \
  --dtype float16 \
  --tp_size 1 --pp_size 1 \
  --use_weight_only \
  --weight_only_precision int4

# Output: trt_checkpoints/gemma3-12b-legal/rank0.safetensors (~7GB)
```

**Phase 3: Build TensorRT engine (.plan) + PTX kernels (~45-90 min)**
```bash
# CRITICAL: Close Ollama first — builder needs all 8GB VRAM temporarily
pkill ollama 2>/dev/null; sleep 2; nvidia-smi  # Verify VRAM free

trtllm-build \
  --checkpoint_dir ./trt_checkpoints/gemma3-12b-legal \
  --output_dir ./trt_engines/gemma3-12b-legal-int4 \
  --gemm_plugin float16 \
  --gpt_attention_plugin float16 \
  --use_weight_only \
  --weight_only_precision int4 \
  --max_batch_size 2 \
  --max_input_len 2048 \
  --max_output_len 512 \
  --max_beam_width 1 \
  --builder_opt 4 \
  --strongly_typed \
  --context_fmha enable \
  --remove_input_padding enable \
  --paged_kv_cache enable \
  --enable_context_fmha_fp32_acc \
  --multi_block_mode enable \
  --use_paged_context_fmha enable \
  --cuda_graph_mode enable \
  --save_pretiming_cache_to_disk enable \
  --load_pretiming_cache_from_disk enable

# Optimization flags explained:
#   --builder_opt 4                  → Max TensorRT optimization level
#   --context_fmha enable            → Fused multi-head attention (Ampere optimized)
#   --paged_kv_cache enable          → PagedAttention for 8GB VRAM
#   --multi_block_mode enable        → Multi-block attention (SM 8.6 CUDA cores)
#   --use_paged_context_fmha enable  → Paged context attention
#   --cuda_graph_mode enable         → CUDA graph capture → replay (eliminates launch overhead)
#   --save_pretiming_cache_to_disk   → Saves compiled .ptx kernel files
#   --strongly_typed                 → Strict type checking (faster kernels)

# Output:
# trt_engines/gemma3-12b-legal-int4/
# ├── rank0.engine       ← .plan file (~6.5 GB)
# ├── config.json
# ├── model.cache        ← Pretiming cache (reuse on rebuild)
# └── ptx/               ← Compiled PTX kernels for SM 8.6
#     ├── gemm_sm86.ptx
#     ├── attention_sm86.ptx
#     ├── activation_sm86.ptx
#     └── layernorm_sm86.ptx

# Verify engine
ls -lh ./trt_engines/gemma3-12b-legal-int4/rank0.engine
# Expected: ~6.5-7 GB

# Quick inference test
trtllm-run \
  --engine_dir ./trt_engines/gemma3-12b-legal-int4 \
  --max_output_len 256 \
  --input_text "Explain Svelte 5 runes and legal evidence classification" \
  --tokenizer_dir ~/gemma3-12b-legal/merged-16bit
# Expected: 120-150 tok/s, first token ~80-120ms
```

**If build fails with "CUDA out of memory":**
```bash
# Reduce optimization level + input length
trtllm-build ... --builder_opt 3 --max_input_len 1024 --max_batch_size 1
```

### Phase 4: WSL2 Docker Triton Deployment (Windows 10 Home)

Deploy the TensorRT engine via Triton Inference Server in Docker on WSL2.

**Prerequisites**: Docker Desktop with WSL2 backend + NVIDIA Container Toolkit installed.

```bash
# ══════════════════════════════════════════════════════════════════
# Step 1: Create Triton model repository
# ══════════════════════════════════════════════════════════════════

mkdir -p ~/triton-models/gemma3_12b_legal/1

# Copy engine + PTX to model directory
cp -r ~/gemma3-12b-legal/TensorRT-LLM/trt_engines/gemma3-12b-legal-int4/* \
      ~/triton-models/gemma3_12b_legal/1/

# Verify structure
ls -lh ~/triton-models/gemma3_12b_legal/1/
# Expected: rank0.engine (~6.5GB), config.json, model.cache, ptx/

# ══════════════════════════════════════════════════════════════════
# Step 2: Create Triton config.pbtxt
# ══════════════════════════════════════════════════════════════════

cat > ~/triton-models/gemma3_12b_legal/config.pbtxt <<'EOF'
name: "gemma3_12b_legal"
backend: "tensorrtllm"
max_batch_size: 2

model_transaction_policy {
  decoupled: True
}

dynamic_batching {
  preferred_batch_size: [1, 2]
  max_queue_delay_microseconds: 100
}

instance_group [
  {
    count: 1
    kind: KIND_GPU
    gpus: [0]
  }
]

parameters: {
  key: "max_beam_width"
  value: { string_value: "1" }
}
parameters: {
  key: "gpt_model_type"
  value: { string_value: "gemma" }
}
parameters: {
  key: "gpt_model_path"
  value: { string_value: "/models/gemma3_12b_legal/1" }
}
parameters: {
  key: "max_tokens_in_paged_kv_cache"
  value: { string_value: "4096" }
}
parameters: {
  key: "batch_scheduler_policy"
  value: { string_value: "max_utilization" }
}
parameters: {
  key: "kv_cache_free_gpu_mem_fraction"
  value: { string_value: "0.85" }
}
parameters: {
  key: "exclude_input_in_output"
  value: { string_value: "true" }
}
parameters: {
  key: "executor_worker_path"
  value: { string_value: "/opt/tritonserver/backends/tensorrtllm/trtllmExecutorWorker" }
}
EOF

# ══════════════════════════════════════════════════════════════════
# Step 3: Pull + Run Triton Docker container
# ══════════════════════════════════════════════════════════════════

# Pull Triton with TRT-LLM backend
docker pull nvcr.io/nvidia/tritonserver:24.02-trtllm-python-py3

# Stop any existing instance
docker stop triton-gemma3-12b 2>/dev/null; docker rm triton-gemma3-12b 2>/dev/null

# Launch Triton (port 8099 = HTTP, 8100 = gRPC, 8101 = Prometheus metrics)
docker run -d --gpus all \
  --name triton-gemma3-12b \
  --shm-size=4g \
  --ulimit memlock=-1 \
  --ulimit stack=67108864 \
  -p 8099:8000 \
  -p 8100:8001 \
  -p 8101:8002 \
  -v ~/triton-models:/models \
  nvcr.io/nvidia/tritonserver:24.02-trtllm-python-py3 \
  tritonserver \
    --model-repository=/models \
    --backend-config=tensorrtllm,max_beam_width=1 \
    --backend-config=tensorrtllm,batch_scheduler_policy=max_utilization \
    --log-verbose=1

# Watch startup logs (wait for "Started HTTPService")
docker logs -f triton-gemma3-12b

# ══════════════════════════════════════════════════════════════════
# Step 4: Verify deployment
# ══════════════════════════════════════════════════════════════════

# Health check
curl http://localhost:8099/v2/health/ready
# Expected: HTTP 200

# Model ready
curl http://localhost:8099/v2/models/gemma3_12b_legal/ready
# Expected: HTTP 200

# Prometheus metrics (throughput, latency, queue depth)
curl http://localhost:8101/metrics | grep nv_inference

# ══════════════════════════════════════════════════════════════════
# Step 5: Test inference
# ══════════════════════════════════════════════════════════════════

pip install tritonclient[http]

python3 -c "
import tritonclient.http as httpclient
import numpy as np, time

client = httpclient.InferenceServerClient(url='localhost:8099')
print('Server ready:', client.is_server_ready())
print('Model ready:', client.is_model_ready('gemma3_12b_legal'))

# Simple inference test
input_ids = np.array([[1, 2, 3, 4, 5]], dtype=np.int32)
inputs = [
    httpclient.InferInput('input_ids', input_ids.shape, 'INT32'),
    httpclient.InferInput('input_lengths', [1], 'INT32'),
    httpclient.InferInput('request_output_len', [1], 'INT32'),
]
inputs[0].set_data_from_numpy(input_ids)
inputs[1].set_data_from_numpy(np.array([5], dtype=np.int32))
inputs[2].set_data_from_numpy(np.array([256], dtype=np.int32))
outputs = [httpclient.InferRequestedOutput('output_ids')]

t0 = time.time()
response = client.infer('gemma3_12b_legal', inputs, outputs=outputs)
print(f'Inference: {time.time()-t0:.3f}s')
print('Output shape:', response.as_numpy('output_ids').shape)
"

# ══════════════════════════════════════════════════════════════════
# Step 6: Auto-start on boot (optional)
# ══════════════════════════════════════════════════════════════════

# Add to Docker restart policy
docker update --restart unless-stopped triton-gemma3-12b

# Or add to WSL2 startup (~/.bashrc or /etc/wsl.conf)
echo 'docker start triton-gemma3-12b 2>/dev/null' >> ~/.bashrc
```

**Ports**: `8099` (HTTP API), `8100` (gRPC), `8101` (Prometheus metrics)
**VRAM**: ~6.5 GB loaded, ~7.5 GB peak (batch 2)
**Latency**: First token 80-120ms, subsequent 15-25ms

### Phase 5: SvelteKit Integration + GPU Arbiter

Wire Triton into the existing Legal AI platform. Your codebase already has the TRT-LLM client + GPU arbiter + LLM router — just update the config.

**SvelteKit `.env`**:
```bash
TENSORRT_SERVICE_URL=http://localhost:8099
```

**Existing files that connect automatically**:
- `src/lib/server/trt-llm/client.ts` — HTTP client for Triton (streaming + non-streaming)
- `src/lib/server/llm/router.ts` — Auto-fallback: Triton → Ollama → Gemini
- `src/lib/server/llm/gpu-arbiter.ts` — Redis VRAM mutex (Ollama vs Triton exclusive lease)
- `src/routes/api/health/capabilities/+server.ts` — Health check includes Triton status

**LLM Router fallback chain** (automatic):
```
User Query → LLM Router
  ├─ Try Triton (port 8099) → 120-150 tok/s ← FASTEST
  ├─ Fallback: Ollama (port 11434) → 60-70 tok/s
  └─ Fallback: Gemini (cloud API) → variable
```

**GPU Arbiter** manages VRAM exclusivity:
- Triton loaded → Ollama 12B model unloaded (can't both fit in 8GB)
- Triton down → Ollama auto-loads gemma3-legal:latest
- Redis lock key: `gpu:tensorrt:lease` (5 min TTL)

---

### Performance Summary — RTX 3060 Ti (8GB VRAM)

| Metric | Path A: Ollama Q4_K_M | Path B: TensorRT INT4 |
|---|---|---|
| **Throughput** | 60-70 tok/s | 120-150 tok/s |
| **First token** | 150-200 ms | 80-120 ms |
| **Subsequent tokens** | 15-20 ms | 10-15 ms |
| **VRAM usage** | 7.2 GB | 6.5 GB |
| **Max batch** | 1 | 2 |
| **Max context** | 4096 tokens | 2048 tokens |
| **Setup time** | 5 minutes | 2-4 hours |

### Full Data Flow (End-to-End)
```
User Query (SvelteKit frontend)
  ↓ SSE via /api/sse/chat
LLM Router (Triton → Ollama → Gemini fallback)
  ↓ GPU Arbiter (Redis VRAM mutex)
TensorRT Engine (rank0.engine, INT4, .ptx SM 8.6)
  ↓ CUDA Graph replay (eliminates kernel launch overhead)
  ↓ PagedAttention (8GB VRAM-efficient KV cache)
  ↓ Fused Multi-Head Attention (Ampere Tensor Cores)
Generated tokens → SSE stream → Frontend
  ↓
Embedding: embeddinggemma → 768-dim vectors
  ↓
Storage: pgvector + Qdrant (dual vector search)
  ↓
Cache: Redis (L3) → Memory (L2) → IndexedDB (L1) → LokiJS (L0)
```

### Troubleshooting
| Issue | Solution |
|---|---|
| "CUDA out of memory" build | `pkill ollama`, reduce `--builder_opt 3 --max_input_len 1024` |
| "CUDA out of memory" inference | Reduce `max_batch_size: 1` in config.pbtxt |
| Triton won't start | `docker run --gpus all nvidia/cuda:12.0-base nvidia-smi` |
| Slow inference (<60 tok/s) | Check `nvidia-smi` — wrong precision or CPU fallback |
| PTX kernels missing | Re-build with `--save_pretiming_cache_to_disk enable` |
| Shards incomplete | Re-download failed shard from Google Drive (others cached) |
| Engine file not found | Check `model.plan` symlink in Triton model repository |

### Reference Documentation
- `scripts/unsloth-training/RTX_3060_TI_TRT_BUILD.md` — Complete build guide (both paths)
- `scripts/unsloth-training/TENSORRT_TRITON_DEPLOYMENT.md` — 7-phase deployment with code
- `scripts/unsloth-training/DEPLOYMENT_ROADMAP.md` — 5-step pipeline overview
- `scripts/unsloth-training/COLAB_PACKAGE/TRAINING_DATA_SUMMARY.md` — Dataset details

## 10. Phase 2 — Emotion + Personality + VLM LoRA (Separate Session)

**Run this AFTER Phase 1 completes and you've saved the merged model.**

Phase 1 trained legal + code knowledge with vision layers FROZEN. Phase 2 adds:
- **Text emotion detection** (28 emotions from GoEmotions)
- **Empathetic personality** (25K conversations from Facebook)
- **Persona consistency** (139K personality-grounded dialogues)
- **Facial emotion recognition** (262K face images with 40 emotion categories)
- **Visual emotion narration** (13K multi-turn dialogues with per-utterance emotions)
- **Document VQA** (16K document image Q&A — legal filings, forms, exhibits)
- **Chart/diagram understanding** (32K chart reasoning — timelines, financial exhibits)
- **UI/screenshot understanding** (1.2K GUI instructions — SvelteKit workspace comprehension)
- **Face instruction following** (1M face+instruction pairs — facial expression + action units)

**Strategy**: Load the Phase 1 merged model, add a NEW LoRA adapter with `finetune_vision_layers=True`, train on emotion/personality/vision data. The base legal knowledge is preserved — we're stacking a new adapter on top.

| Dataset | Repo | Examples | Purpose |
|---------|------|----------|---------|
| GoEmotions | `google-research-datasets/go_emotions` | 58K | 28-class text emotion (multi-label) |
| Empathetic Dialogues | `facebook/empathetic_dialogues` | 25K convos | Empathetic response generation |
| PersonaChat | `bavard/personachat_truecased` | 139K | 4-trait personality consistency |
| EmoNet-Face | `laion/emonet-face-big` | 262K | 40-category facial emotion + VLM reasoning |
| DailyDialog | `li2017dailydialog/daily_dialog` | 13K | Per-utterance emotion in conversation |
| Emotion Sentiment | `VaisakhKrishna/Emotional_Sentiment_Analysis` | 7K | Pre-formatted empathetic responses |
| **DocVQA** | `lmms-lab/DocVQA` | 16K | Document image Q&A (scans, forms, exhibits) |
| **ChartQA** | `ahmed-masry/ChartQA` | 32K | Chart/timeline/financial exhibit reasoning |
| **ScreenSpot** | `OS-Copilot/ScreenSpot` | 1.2K | UI screenshot understanding (buttons, fields) |
| **FaceInstruct1M** | `chaubeyG/FaceInstruct1M` | 1M | Face + instruction/response (expressions, action units) |

**Requirements**: A100 40GB (vision layers need extra VRAM), ~3-4 hours training

In [ ]:
# ══════════════════════════════════════════════════════════════════
# Phase 2: Load emotion + personality + vision datasets
# ══════════════════════════════════════════════════════════════════

EMOTION_DATASETS = {}

# 1. GoEmotions — 28-class multi-label text emotion (Google)
print("[1/10] GoEmotions (28 emotions)...", end=" ")
go_emotions = load_dataset("google-research-datasets/go_emotions", "simplified", split="train")
emotion_names = ['admiration', 'amusement', 'anger', 'annoyance', 'approval', 'caring',
    'confusion', 'curiosity', 'desire', 'disappointment', 'disapproval', 'disgust',
    'embarrassment', 'excitement', 'fear', 'gratitude', 'grief', 'joy', 'love',
    'nervousness', 'optimism', 'pride', 'realization', 'relief', 'remorse',
    'sadness', 'surprise', 'neutral']

def format_goemotion(ex):
    labels = [emotion_names[i] for i in ex['labels']]
    label_str = ', '.join(labels) if labels else 'neutral'
    return {"conversations": [
        {"from": "human", "value": f"Detect the emotions in this text: \"{ex['text']}\""},
        {"from": "gpt", "value": f"Emotions detected: {label_str}.\n\nThe text expresses {label_str}. "
         f"{'The speaker seems to feel multiple emotions simultaneously.' if len(labels) > 1 else ''}"}
    ]}

EMOTION_DATASETS['goemotions'] = go_emotions.map(format_goemotion, remove_columns=go_emotions.column_names, num_proc=4)
print(f"{len(EMOTION_DATASETS['goemotions']):,}")

# 2. Facebook Empathetic Dialogues — empathetic response training
print("[2/10] Empathetic Dialogues...", end=" ")
emp_dialogues = load_dataset("facebook/empathetic_dialogues", split="train")

def format_empathetic(ex):
    context = ex.get('situation', '') or ex.get('context', '')
    utterance = ex.get('utterance', '')
    emotion = ex.get('context', '')  # emotion context label
    return {"conversations": [
        {"from": "human", "value": f"[Emotion context: {emotion}] {context}"},
        {"from": "gpt", "value": utterance}
    ]}

EMOTION_DATASETS['empathetic'] = emp_dialogues.map(format_empathetic, remove_columns=emp_dialogues.column_names, num_proc=4)
print(f"{len(EMOTION_DATASETS['empathetic']):,}")

# 3. PersonaChat — personality-consistent conversation
print("[3/10] PersonaChat (personality traits)...", end=" ")
personachat = load_dataset("bavard/personachat_truecased", split="train")

def format_persona(ex):
    persona = ' | '.join(ex.get('personality', [])[:4])
    history = ex.get('history', [])
    candidates = ex.get('candidates', [])
    response = candidates[-1] if candidates else ''
    context = history[-1] if history else 'Hello!'
    return {"conversations": [
        {"from": "human", "value": f"[Your personality: {persona}]\n\nUser: {context}"},
        {"from": "gpt", "value": response}
    ]}

EMOTION_DATASETS['personachat'] = personachat.map(format_persona, remove_columns=personachat.column_names, num_proc=4)
print(f"{len(EMOTION_DATASETS['personachat']):,}")

# 4. DailyDialog — per-utterance emotion in multi-turn conversation
print("[4/10] DailyDialog (conversation emotion)...", end=" ")
daily_dialog = load_dataset("li2017dailydialog/daily_dialog", split="train")
dd_emotions = ['no_emotion', 'anger', 'disgust', 'fear', 'happiness', 'sadness', 'surprise']

def format_daily(ex):
    dialog = ex.get('dialog', [])
    emotions = ex.get('emotion', [])
    if len(dialog) < 2:
        return {"conversations": [{"from": "human", "value": "Hello"}, {"from": "gpt", "value": "Hi there!"}]}
    user_turn = dialog[0]
    assistant_turn = dialog[1]
    emotion_label = dd_emotions[emotions[1]] if len(emotions) > 1 and emotions[1] < len(dd_emotions) else 'neutral'
    return {"conversations": [
        {"from": "human", "value": user_turn},
        {"from": "gpt", "value": f"[feeling: {emotion_label}] {assistant_turn}"}
    ]}

EMOTION_DATASETS['dailydialog'] = daily_dialog.map(format_daily, remove_columns=daily_dialog.column_names, num_proc=4)
print(f"{len(EMOTION_DATASETS['dailydialog']):,}")

# 5. Emotional Sentiment Analysis — pre-formatted empathetic responses
print("[5/10] Emotional Sentiment Analysis...", end=" ")
try:
    emo_sentiment = load_dataset("VaisakhKrishna/Emotional_Sentiment_Analysis", split="train")
    def format_emo_sent(ex):
        instruction = ex.get('instruction', ex.get('input', ''))
        output = ex.get('output', ex.get('response', ''))
        return {"conversations": [
            {"from": "human", "value": instruction},
            {"from": "gpt", "value": output}
        ]}
    EMOTION_DATASETS['emo_sentiment'] = emo_sentiment.map(format_emo_sent, remove_columns=emo_sentiment.column_names, num_proc=4)
    print(f"{len(EMOTION_DATASETS['emo_sentiment']):,}")
except Exception as e:
    print(f"SKIPPED ({e})")

# 6. boltuix emotions — 13-class including confusion/sarcasm (MIT license)
print("[6/10] boltuix emotions (13 classes)...", end=" ")
try:
    boltuix = load_dataset("boltuix/emotions-dataset", split="train")
    def format_boltuix(ex):
        return {"conversations": [
            {"from": "human", "value": f"What emotion is expressed? \"{ex.get('text', '')}\""},
            {"from": "gpt", "value": f"The emotion expressed is: {ex.get('label', 'neutral')}"}
        ]}
    EMOTION_DATASETS['boltuix'] = boltuix.map(format_boltuix, remove_columns=boltuix.column_names, num_proc=4)
    print(f"{len(EMOTION_DATASETS['boltuix']):,}")
except Exception as e:
    print(f"SKIPPED ({e})")

# ── Vision Datasets (SigLIP fine-tuning — Phase 2 unfreezes vision layers) ──

from PIL import Image
import io, base64

# 7. DocVQA — document image question answering (scans, forms, legal filings)
print("[7/10] DocVQA (document VQA)...", end=" ")
try:
    docvqa = load_dataset("lmms-lab/DocVQA", split="train[:5000]")
    docvqa_formatted = []
    for ex in docvqa:
        img = ex.get('image')
        question = ex.get('question', '')
        answers = ex.get('answers', [])
        answer = answers[0] if answers else 'Unable to determine.'
        if img is not None and question:
            buf = io.BytesIO()
            img.save(buf, format='PNG')
            img_b64 = base64.b64encode(buf.getvalue()).decode('utf-8')
            docvqa_formatted.append({"conversations": [
                {"from": "human", "value": [
                    {"type": "image", "image": img_b64},
                    {"type": "text", "text": question}
                ]},
                {"from": "gpt", "value": str(answer)}
            ]})
    EMOTION_DATASETS['docvqa'] = Dataset.from_list(docvqa_formatted)
    print(f"{len(docvqa_formatted):,} document VQA pairs")
except Exception as e:
    print(f"SKIPPED ({e})")

# 8. ChartQA — chart/timeline/financial exhibit reasoning
print("[8/10] ChartQA (chart reasoning)...", end=" ")
try:
    chartqa = load_dataset("ahmed-masry/ChartQA", split="train[:5000]")
    chartqa_formatted = []
    for ex in chartqa:
        img = ex.get('image')
        question = ex.get('query', ex.get('question', ''))
        answer = ex.get('label', ex.get('answer', ''))
        if img is not None and question:
            buf = io.BytesIO()
            img.save(buf, format='PNG')
            img_b64 = base64.b64encode(buf.getvalue()).decode('utf-8')
            chartqa_formatted.append({"conversations": [
                {"from": "human", "value": [
                    {"type": "image", "image": img_b64},
                    {"type": "text", "text": question}
                ]},
                {"from": "gpt", "value": str(answer)}
            ]})
    EMOTION_DATASETS['chartqa'] = Dataset.from_list(chartqa_formatted)
    print(f"{len(chartqa_formatted):,} chart reasoning pairs")
except Exception as e:
    print(f"SKIPPED ({e})")

# 9. ScreenSpot — UI/screenshot/desktop understanding (buttons, fields, browser UI)
print("[9/10] ScreenSpot (UI screenshots)...", end=" ")
try:
    screenspot = load_dataset("OS-Copilot/ScreenSpot", split="train")
    screenspot_formatted = []
    for ex in screenspot:
        img = ex.get('image')
        instruction = ex.get('instruction', ex.get('text', ''))
        if img is not None and instruction:
            buf = io.BytesIO()
            img.save(buf, format='PNG')
            img_b64 = base64.b64encode(buf.getvalue()).decode('utf-8')
            # ScreenSpot has bounding box targets — format as location description
            bbox = ex.get('bbox', [])
            if bbox and len(bbox) == 4:
                answer = f"The element is located at coordinates [{bbox[0]:.0f}, {bbox[1]:.0f}, {bbox[2]:.0f}, {bbox[3]:.0f}]."
            else:
                answer = "Element identified in the screenshot."
            screenspot_formatted.append({"conversations": [
                {"from": "human", "value": [
                    {"type": "image", "image": img_b64},
                    {"type": "text", "text": instruction}
                ]},
                {"from": "gpt", "value": answer}
            ]})
    EMOTION_DATASETS['screenspot'] = Dataset.from_list(screenspot_formatted)
    print(f"{len(screenspot_formatted):,} UI instruction pairs")
except Exception as e:
    print(f"SKIPPED ({e})")

# 10. FaceInstruct1M — face + instruction/response (expressions, facial action units)
print("[10/10] FaceInstruct1M (face instruction)...", end=" ")
try:
    # Load a subset — full dataset is 1M examples
    faceinst = load_dataset("chaubeyG/FaceInstruct1M", split="train[:10000]")
    faceinst_formatted = []
    for ex in faceinst:
        img = ex.get('image')
        instruction = ex.get('instruction', ex.get('question', ''))
        response = ex.get('response', ex.get('answer', ''))
        if img is not None and instruction and response:
            buf = io.BytesIO()
            img.save(buf, format='PNG')
            img_b64 = base64.b64encode(buf.getvalue()).decode('utf-8')
            faceinst_formatted.append({"conversations": [
                {"from": "human", "value": [
                    {"type": "image", "image": img_b64},
                    {"type": "text", "text": instruction}
                ]},
                {"from": "gpt", "value": str(response)}
            ]})
    EMOTION_DATASETS['faceinst'] = Dataset.from_list(faceinst_formatted)
    print(f"{len(faceinst_formatted):,} face instruction pairs")
except Exception as e:
    print(f"SKIPPED ({e})")

# Combine all emotion + vision datasets
from unsloth.chat_templates import standardize_sharegpt
emotion_combined = concatenate_datasets(list(EMOTION_DATASETS.values()))
emotion_combined = standardize_sharegpt(emotion_combined)

# Filter empty
emotion_combined = emotion_combined.filter(
    lambda ex: any(len((c.get("value") or c.get("content") or "").strip()) > 0 for c in ex["conversations"]),
    num_proc=4,
)

print(f"\n{'='*60}")
print(f"Phase 2 dataset: {len(emotion_combined):,} examples")
print(f"  Text emotion datasets:")
for name in ['goemotions', 'empathetic', 'personachat', 'dailydialog', 'emo_sentiment', 'boltuix']:
    if name in EMOTION_DATASETS:
        print(f"    {name}: {len(EMOTION_DATASETS[name]):,}")
print(f"  Vision datasets (SigLIP fine-tuning):")
for name in ['docvqa', 'chartqa', 'screenspot', 'faceinst']:
    if name in EMOTION_DATASETS:
        print(f"    {name}: {len(EMOTION_DATASETS[name]):,}")

In [ ]:
# ══════════════════════════════════════════════════════════════════
# Phase 2: Load Phase 1 model + add emotion LoRA (VISION UNFROZEN)
# ══════════════════════════════════════════════════════════════════

# Option A: Load from Phase 1 merged model (if in same session)
# Option B: Load from Google Drive / HuggingFace (new session)

PHASE1_MODEL = "gemma3-12b-legal-merged-16bit"  # From Phase 1 export (cell 23)
# PHASE1_MODEL = "/content/drive/MyDrive/gemma3-12b-legal-shards"  # If loading from Drive

import os
if not os.path.exists(PHASE1_MODEL):
    print(f"Phase 1 model not found at {PHASE1_MODEL}")
    print("Loading base model instead (legal knowledge from Phase 1 LoRA will be in merged weights)")
    PHASE1_MODEL = MODEL_NAME  # Fall back to base model

print(f"Loading Phase 1 model: {PHASE1_MODEL}...")
model_p2, tokenizer_p2 = FastVisionModel.from_pretrained(
    model_name=PHASE1_MODEL,
    max_seq_length=1024,  # Longer context for conversations
    load_in_4bit=True,
    dtype=None,
)
tokenizer_p2 = get_chat_template(tokenizer_p2, chat_template="gemma-3")

# Add NEW LoRA adapter — VISION LAYERS UNFROZEN for facial emotion
print("\nAdding emotion LoRA adapter (vision UNFROZEN)...")
model_p2 = FastVisionModel.get_peft_model(
    model_p2,
    r=8,                # Lower rank (emotion is simpler than legal reasoning)
    lora_alpha=16,
    lora_dropout=0.05,
    finetune_vision_layers=True,     # UNFROZEN — learns facial emotion from images
    finetune_language_layers=True,    # Emotion text understanding
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    use_gradient_checkpointing=True,
    use_rslora=True,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    random_state=42,
)
model_p2.print_trainable_parameters()
print("\nVision layers: TRAINABLE | Emotion adapter ready")

In [ ]:
# ══════════════════════════════════════════════════════════════════
# Phase 2: Train emotion + personality LoRA
# ══════════════════════════════════════════════════════════════════
from trl import SFTConfig
from unsloth.chat_templates import train_on_responses_only

emotion_config = SFTConfig(
    output_dir="./gemma3-12b-emotion-outputs",
    num_train_epochs=2,
    per_device_train_batch_size=1,            # Vision layers need more VRAM
    gradient_accumulation_steps=16,           # Effective batch: 1×16=16
    learning_rate=1e-4,                       # Lower LR — fine-tuning on top of fine-tune
    warmup_steps=100,
    fp16=False, bf16=True, bf16_full_eval=True,
    logging_steps=25,
    save_strategy="steps", save_steps=500, save_total_limit=2,
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    max_grad_norm=1.0,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": True},
    dataloader_num_workers=4,
    dataloader_pin_memory=True,
    seed=42, data_seed=42,
    report_to="wandb" if USE_WANDB else "none",
    max_seq_length=1024,
    packing=False,
)

def emotion_formatting_func(examples):
    convos = examples["conversations"]
    if isinstance(convos, list) and len(convos) > 0 and isinstance(convos[0], dict):
        return [tokenizer_p2.apply_chat_template(convos, tokenize=False, add_generation_prompt=False)]
    return [tokenizer_p2.apply_chat_template(c, tokenize=False, add_generation_prompt=False) for c in convos]

emotion_trainer = SFTTrainer(
    model=model_p2, tokenizer=tokenizer_p2, train_dataset=emotion_combined,
    args=emotion_config,
    formatting_func=emotion_formatting_func,
)

# Mask user prompts — only train on assistant (model) outputs
emotion_trainer = train_on_responses_only(
    emotion_trainer,
    instruction_part="<start_of_turn>user\n",
    response_part="<start_of_turn>model\n",
)

print(f"Phase 2 trainer ready | {len(emotion_combined):,} examples | Vision: UNFROZEN | Responses-only")
print(f"Checkpoints: every {emotion_config.save_steps} steps (keep {emotion_config.save_total_limit})")

# Train (auto-resumes from checkpoint if Colab reconnects)
checkpoint_dirs_p2 = sorted(glob.glob("./gemma3-12b-emotion-outputs/checkpoint-*"))
if checkpoint_dirs_p2:
    print(f"Resuming Phase 2 from: {checkpoint_dirs_p2[-1]}")
    emotion_stats = emotion_trainer.train(resume_from_checkpoint=True)
else:
    print(f"Starting Phase 2 fresh — {len(emotion_combined):,} examples, ~3-4 hours on A100...")
    emotion_stats = emotion_trainer.train()

rt = emotion_stats.metrics['train_runtime']
print(f"\nPhase 2 done: {rt/3600:.1f}h | {emotion_stats.metrics['train_samples_per_second']:.2f} samples/sec")

In [ ]:
# ══════════════════════════════════════════════════════════════════
# Phase 2: Test emotion detection + empathetic responses
# ══════════════════════════════════════════════════════════════════
FastVisionModel.for_inference(model_p2)
text_streamer_p2 = TextStreamer(tokenizer_p2, skip_prompt=True)

emotion_prompts = [
    # Text emotion detection
    'Detect the emotions in this text: "I can\'t believe they dismissed my case after all that work. This is so unfair."',
    # Empathetic response
    "[Emotion context: frustrated] The client just received bad news about their appeal being denied.",
    # Personality-grounded
    "[Your personality: I am patient | I explain complex topics simply | I care about people]\n\nUser: I don't understand what habeas corpus means and I'm scared.",
    # Legal + emotion combined
    "A witness is crying on the stand while describing the accident. How should the attorney respond?",
]

for prompt in emotion_prompts:
    print(f"\n{'='*70}\n{prompt[:80]}...\n{'='*70}")
    inputs = tokenizer_p2.apply_chat_template(
        [{"role": "user", "content": prompt}],
        tokenize=True, add_generation_prompt=True, return_tensors="pt",
    ).to("cuda")
    model_p2.generate(input_ids=inputs, streamer=text_streamer_p2,
                      max_new_tokens=256, temperature=0.7, top_p=0.9, use_cache=True)
    print()

In [ ]:
# ══════════════════════════════════════════════════════════════════
# Phase 2: Save + Export emotion LoRA
# ══════════════════════════════════════════════════════════════════

# Save emotion LoRA adapter separately
model_p2.save_pretrained("gemma3-12b-emotion-lora")
tokenizer_p2.save_pretrained("gemma3-12b-emotion-lora")
print("Emotion LoRA saved: gemma3-12b-emotion-lora/ (~300 MB)")

# Export merged model (legal + emotion combined)
EMOTION_EXPORT = Path("gemma3-12b-legal-emotion-merged-16bit")
print(f"\nExporting merged model (legal + emotion + vision)...")
model_p2.save_pretrained_merged(
    str(EMOTION_EXPORT), tokenizer_p2, save_method="merged_16bit",
)

shard_files = sorted(EMOTION_EXPORT.glob("model-*.safetensors"))
if not shard_files:
    shard_files = sorted(EMOTION_EXPORT.glob("*.safetensors"))
total_size = sum(f.stat().st_size for f in EMOTION_EXPORT.iterdir()) / (1024**3)
print(f"Export complete: {len(shard_files)} shards, {total_size:.1f} GB")

# Copy to Google Drive (reuses Phase 1 cache-safe pattern)
EMOTION_DRIVE_DIR = Path("/content/drive/MyDrive/gemma3-12b-legal-emotion-shards")
EMOTION_DRIVE_DIR.mkdir(parents=True, exist_ok=True)

import shutil, time
all_files = sorted(EMOTION_EXPORT.iterdir())
copied, skipped = 0, 0
for i, src in enumerate(all_files, 1):
    dst = EMOTION_DRIVE_DIR / src.name
    if dst.exists() and dst.stat().st_size == src.stat().st_size:
        skipped += 1; continue
    shutil.copy2(str(src), str(dst))
    copied += 1

print(f"\nGoogle Drive: {copied} copied, {skipped} skipped → {EMOTION_DRIVE_DIR}")
print(f"\nPhase 2 complete! Model now has:")
print(f"  - Legal reasoning (Phase 1: 81K legal + code examples)")
print(f"  - 28-class text emotion detection (GoEmotions)")
print(f"  - Empathetic response generation (Facebook)")
print(f"  - Personality consistency (PersonaChat)")
print(f"  - Conversation emotion tracking (DailyDialog)")
print(f"  - Vision: Document VQA (DocVQA — legal filings, forms)")
print(f"  - Vision: Chart reasoning (ChartQA — timelines, financial exhibits)")
print(f"  - Vision: UI understanding (ScreenSpot — app screenshots, buttons)")
print(f"  - Vision: Face instruction (FaceInstruct1M — expressions, action units)")
print(f"  - SigLIP 400M encoder TRAINED (vision layers unfrozen)")
print(f"\nDeploy with same Path A/B from Phase 1 — just use the emotion-merged shards.")
print(f"Single multimodal engine handles text + vision + emotion. No separate VLM endpoint needed.")